# ChakraModel Eval Harness — Kaggle Runner

Requires:
1. Dataset **ChakraModel_EvalHarness_Code** (the tiny code zip) attached
2. Dataset **ChakraModel_EvalHarness_Weights** (chakra_transformer_best.pth + combo1_best.pth) attached
3. Any polyp datasets you want scored (PolypGen, HyperKvasir, CVC-ClinicDB, etc.) attached — no code changes needed, `dataset_discovery.py` auto-scans `/kaggle/input`
4. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (timm needs to fetch the ViT-Large pretrained backbone once before the checkpoint overwrites it), **Persistence = Files only** if you want outputs kept between sessions

In [1]:
!pip install -q timm albumentations opencv-python-headless psutil

In [2]:
import os, shutil, glob

WORK = '/kaggle/working'

# --- get the code, either as a zip (rare — Kaggle usually auto-extracts
#     zip uploads on ingest) or already-extracted under /kaggle/input ---
code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code.zip', recursive=True):
    code_zip = p
    break

if code_zip:
    print('Unpacking code from', code_zip)
    shutil.unpack_archive(code_zip, WORK)
else:
    # Kaggle auto-extracted the zip on upload — find the already-unpacked
    # eval_harness/ folder anywhere under /kaggle/input and copy it (and
    # its sibling src/) into /kaggle/working instead.
    eh_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'eval_harness' and 'run_eval.py' in files:
            eh_dir = root
            break
    assert eh_dir, (
        'Neither ChakraModel_EvalHarness_Code.zip nor an extracted eval_harness/ '
        'folder was found under /kaggle/input — attach that dataset.'
    )
    dataset_root = os.path.dirname(eh_dir)
    print('Found already-extracted code at', dataset_root, '(Kaggle auto-extracted the zip on upload)')
    shutil.copytree(eh_dir, f'{WORK}/eval_harness', dirs_exist_ok=True)
    src_dir = os.path.join(dataset_root, 'src')
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f'{WORK}/src', dirs_exist_ok=True)
    print('Copied eval_harness/ and src/ into', WORK)

# --- stage weights where run_eval.py's defaults expect them ---
os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing, '— attach the ChakraModel_EvalHarness_Weights dataset.')
else:
    print('Both checkpoints staged OK.')

Found already-extracted code at /kaggle/input/datasets/jayasrikannuchamy/chakramodel-evalharness-code (Kaggle auto-extracted the zip on upload)
Copied eval_harness/ and src/ into /kaggle/working
Linked combo1_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/combo1_best.pth
Linked chakra_transformer_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/chakra_transformer_best.pth
Both checkpoints staged OK.


In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
 - Tesla T4
 - Tesla T4


In [4]:
import os

eval_script_path = '/kaggle/working/eval_harness/run_eval.py'
if os.path.exists(eval_script_path):
    with open(eval_script_path, 'r') as f:
        content = f.read()
    
    # Replace high worker counts with 4
    new_content = content.replace('num_workers=16', 'num_workers=4')
    new_content = new_content.replace('num_workers = 16', 'num_workers = 4')
    
    with open(eval_script_path, 'w') as f:
        f.write(new_content)
    print("Successfully updated num_workers to 4 in run_eval.py!")
else:
    print("run_eval.py not found yet. Make sure the setup cell has run to unpack the code.")

Successfully updated num_workers to 4 in run_eval.py!


In [5]:
import os, shutil

os.makedirs('/kaggle/working/weights/checkpoints', exist_ok=True)
weights_to_copy = ['chakra_transformer_best.pth', 'combo1_best.pth']

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in weights_to_copy:
            src_path = os.path.join(root, f)
            dst_path = os.path.join('/kaggle/working/weights/checkpoints', f)
            shutil.copy(src_path, dst_path)
            print(f"Staged {f} successfully!")

print("Checkpoints directory contents:", os.listdir('/kaggle/working/weights/checkpoints'))

Staged combo1_best.pth successfully!
Staged chakra_transformer_best.pth successfully!
Checkpoints directory contents: ['combo1_best.pth', 'chakra_transformer_best.pth']


In [6]:
import os

# 1. Patch run_eval.py worker count
eval_script_path = '/kaggle/working/eval_harness/run_eval.py'
if os.path.exists(eval_script_path):
    with open(eval_script_path, 'r') as f:
        content = f.read()
    
    # Replace high worker counts with 4
    new_content = content.replace('num_workers=16', 'num_workers=4')
    new_content = new_content.replace('num_workers = 16', 'num_workers = 4')
    
    with open(eval_script_path, 'w') as f:
        f.write(new_content)
    print("Successfully updated num_workers to 4 in run_eval.py!")
else:
    print("Error: run_eval.py not found. Make sure setup has run.")

# 2. Patch eval_engine.py worker count if hardcoded there too
engine_path = '/kaggle/working/eval_harness/eval_engine.py'
if os.path.exists(engine_path):
    with open(engine_path, 'r') as f:
        eng_content = f.read()
    eng_content = eng_content.replace('num_workers=16', 'num_workers=4')
    eng_content = eng_content.replace('num_workers = 16', 'num_workers = 4')
    with open(engine_path, 'w') as f:
        f.write(eng_content)
    print("Successfully patched eval_engine.py worker count as well!")

Successfully updated num_workers to 4 in run_eval.py!
Successfully patched eval_engine.py worker count as well!


In [7]:
import os

eval_script_path = '/kaggle/working/eval_harness/run_eval.py'
with open(eval_script_path, 'r') as f:
    code = f.read()

# Force num_workers to be capped at 4 globally in the script
if 'num_workers' in code:
    # Replace common dictionary or config patterns
    code = code.replace("['num_workers'] = 16", "['num_workers'] = 4")
    code = code.replace('["num_workers"] = 16', '["num_workers"] = 4')
    code = code.replace('num_workers=16', 'num_workers=4')
    code = code.replace('num_workers = 16', 'num_workers = 4')
    # If workers are derived from os.cpu_count() or similar logic returning 16
    code = code.replace('16', '4')

with open(eval_script_path, 'w') as f:
    f.write(code)
print("Forced worker count override applied!")

Forced worker count override applied!


In [8]:
import torch, os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

In [9]:
import os

engine_path = '/kaggle/working/eval_harness/eval_engine.py'
if os.path.exists(engine_path):
    with open(engine_path, 'r') as f:
        content = f.read()

    # Lower batch size limits to prevent T4 OOM
    content = content.replace('MAX_BATCH = 128', 'MAX_BATCH = 32')
    content = content.replace('batch_size = 128', 'batch_size = 32')
    content = content.replace('initial_batch = 32', 'initial_batch = 16')

    with open(engine_path, 'w') as f:
        f.write(content)
    print("Batch sizes lowered to prevent OOM!")

Batch sizes lowered to prevent OOM!


In [10]:
%cd /kaggle/working
!python eval_harness/run_eval.py

/kaggle/working
[Engine] Detected 2 GPU(s).
[Engine] ViT-Large → cuda:0 | PraNet → cuda:1
[Engine] Multi-GPU parallel mode: True

  ChakraModel Universal Evaluation Harness — FULL STRESS MODE
  PyTorch   : 2.10.0+cu128
  GPUs      : 2
    cuda:0 = Tesla T4  (14.6 GB VRAM)
    cuda:1 = Tesla T4  (14.6 GB VRAM)
  CPU cores : 4
  RAM       : 31.3 GB
  Strategy  : FP16 autocast | CUDA streams | Aggressive batch ramp
  Throttling: NONE — backs off only at >98% VRAM (OOM protection only)

[Config] cudnn.benchmark=True | TF32 matmul=high | deterministic=False

[Main] Dataset root : /kaggle/input

[Main] Scanning for datasets...
  [DISCOVER] datasets                                 | PAIRED_IMAGE    |   1552 frames

[Main] Total datasets to evaluate: 1
[Main] Total frames              : 1,552

[Main] Loading models...
model.safetensors: 100%|████████████████████| 1.22G/1.22G [00:07<00:00, 153MB/s]

[Engine] Loading ViT-Large from chakra_transformer_best.pth → cuda:0
  [WARN] Missing keys in ch

In [11]:
import os, shutil

# Find where eval_harness is located in /kaggle/input
src_eval_harness = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'run_eval.py' in files:
        src_eval_harness = os.path.dirname(os.path.abspath(root)) if root.endswith('eval_harness') else root
        break

if src_eval_harness:
    shutil.copytree(os.path.join(src_eval_harness, 'eval_harness'), '/kaggle/working/eval_harness', dirs_exist_ok=True)
    if os.path.exists(os.path.join(src_eval_harness, 'src')):
        shutil.copytree(os.path.join(src_eval_harness, 'src'), '/kaggle/working/src', dirs_exist_ok=True)
    print("Successfully copied evaluation code to /kaggle/working!")
else:
    print("Could not locate run_eval.py under /kaggle/input. Please ensure your code dataset is attached.")

Successfully copied evaluation code to /kaggle/working!


In [12]:
# Sanity peek at the results file
import json, glob
res_files = sorted(glob.glob('/kaggle/working/eval_harness/results/evaluation_results_*.json'))
assert res_files, 'No results file produced.'
with open(res_files[-1]) as f:
    results = json.load(f)
for r in results:
    print(r.get('dataset'), '->', r.get('vit_large', {}).get('dice'), r.get('error', ''))

AssertionError: No results file produced.